# SemEval-2026 Task 2 & 3 (Track A: DimABSA)
# Subtask 2: DimASTE - Dimensional Aspect Sentiment Triplet Extraction
# Subtask 3: DimASQE - Dimensional Aspect Sentiment Quadruplets Extraction

-----

## Starter Notebook
LLM-based finetune for Dimensional Aspect Sentiment Triplet (and Quadruplet) Extraction

## Introduction:

You are welcome to participate in our SemEval Shared Task!

In this starter notebook, we guide you through the process of fine-tuning a pre-trained language model on sample training data to build a dimensional sentiment extraction model.  
This notebook is adapted from a HuggingFace-style implementation for similar tasks.

### Outline:
- Installation and importation of necessary libraries Setting up the project parameters. Running training and evaluation Before you start:

- It is strongly advised that you use a GPU to speed up training. To do this, go to the "Runtime" menu in Colab, select "Change runtime type" and then in the popup menu, choose "GPU" in the "Hardware accelerator" box.

### NB:
- This notebook aims to help you become familiar with fine-tuning language models for dimensional sentiment tasks.  
- You are encouraged to extend or modify it to obtain competitive performance.
- This notebook will take about 2 to 2.5 hours to run. It may shut down if your Colab GPU quota is insufficient.

### Languages and Domains:
#### Track A: Subtask 2 & 3
- eng_restaurant
- eng_laptop
- jpn_hotel
- rus_restaurant
- tat_restaurant
- ukr_restaurant
- zho_restaurant
- zho_laptop

### Model:
This Starter Notebook uses the Qwen3-4B-Instruct pretrained language model, adapted through Unsloth for efficient fine-tuning.

The model is a multilingual decoder-based LLM with strong instruction-following ability, and it is provided in bnb 4-bit quantized form, allowing training on limited GPU resources such as Google Colab.

You can find the model here:
https://huggingface.co/unsloth/Qwen3-4B-Instruct-2507-bnb-4bit

If you need alternative models (e.g., larger Qwen variants, or other multilingual LLMs), you may explore them on Hugging Face:
https://huggingface.co/unsloth/models

### Install unsloth package

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Load the competition data

In [ ]:
import os
import json
from datasets import load_dataset

# Task Configuration
subtask = "subtask_2"
task = "task2"
lang = "eng"
domains = ["laptop", "restaurant"]

all_datasets = {}

for domain in domains:
    print(f"Loading data for {domain.upper()}...")

    # URLs
    train_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_train_alltasks.jsonl"
    dev_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_dev_{task}.jsonl"
    test_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_test_{task}.jsonl"

    try:
        train_data = load_dataset("json", data_files=train_url, split="train")
        dev_data = load_dataset("json", data_files=dev_url, split="train")
        test_data = load_dataset("json", data_files=test_url, split="train")

        all_datasets[domain] = {
            "train": train_data,
            "validation": dev_data,
            "test": test_data
        }

        print(f"{domain.capitalize()} successfully loaded!")
        print(f"   - Train (Quadruplets): {len(train_data)} samples")
        print(f"   - Dev (Triplets): {len(dev_data)} samples")
        print(f"   - Test (Extraction): {len(test_data)} samples")

    except Exception as e:
        print(f"Error loading {domain} data: {e}")

Loading data for LAPTOP...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Laptop successfully loaded!
   - Train (Quadruplets): 4076 samples
   - Dev (Triplets): 200 samples
   - Test (Extraction): 1000 samples
Loading data for RESTAURANT...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Restaurant successfully loaded!
   - Train (Quadruplets): 2284 samples
   - Dev (Triplets): 200 samples
   - Test (Extraction): 1000 samples


### Display the data info

In [ ]:
for domain in domains:
    print(f"\n{'-'*60}")
    print(f" DOMAIN: {domain.upper()}")
    print(f"{'-'*60}")

    for split in ['train', 'validation', 'test']:
        ds = all_datasets[domain][split]

        print(f"\n--- {domain.capitalize()} {split.capitalize()} Set ---")
        print(f"Total Samples: {len(ds)}")
        print(f"Columns: {ds.column_names}")
        print(f"First Example:")
        print(json.dumps(ds[0], indent=2))
        print("-" * 30)


------------------------------------------------------------
 DOMAIN: LAPTOP
------------------------------------------------------------

--- Laptop Train Set ---
Total Samples: 4076
Columns: ['ID', 'Text', 'Quadruplet']
First Example:
{
  "ID": "laptop_quad_dev_1",
  "Text": "this unit is ` ` pretty ` ` and stylish , so my high school daughter was attracted to it for that reason .",
  "Quadruplet": [
    {
      "Aspect": "unit",
      "Category": "LAPTOP#DESIGN_FEATURES",
      "Opinion": "pretty",
      "VA": "7.12#7.12"
    },
    {
      "Aspect": "unit",
      "Category": "LAPTOP#DESIGN_FEATURES",
      "Opinion": "stylish",
      "VA": "7.12#7.12"
    }
  ]
}
------------------------------

--- Laptop Validation Set ---
Total Samples: 200
Columns: ['ID', 'Text', 'Triplet']
First Example:
{
  "ID": "lap26_aste_dev_1",
  "Text": "Great perforemce at a great price",
  "Triplet": [
    {
      "Aspect": "perforemce",
      "Opinion": "Great",
      "VA": "6.88#7.25"
    },
    {
 

### Design prompt template

In [ ]:
# Task 2 Unified Prompt Template
instruction = '''Below is an instruction describing a task, paired with an input that provides additional context. Your goal is to generate an output that correctly completes the task.

### Instruction:
Given a textual instance [Text], extract all (A, O, VA) triplets, where:
- A is an Aspect term (the entity being discussed)
- O is an Opinion term (the feeling or sentiment expressed about A)
- VA is a Valence–Arousal score in the format (valence#arousal)

Valence: 1.00 (negative) to 9.00 (positive).
Arousal: 1.00 (calm) to 9.00 (excited).
Important: Aspect and Opinion must maintain the exact case and spelling found in the [Text].

### Example:
Input:
[Text] average to good thai food, but terrible delivery.

Output:
[Triplet] (thai food, average to good, 6.75#6.38), (delivery, terrible, 2.88#6.62)

### Question:
Now complete the following example:
Input:
'''

def convert(x):
    """
    Converts a dataset entry into an instruction-tuned prompt format.
    Handles Train (Quadruplet) and Dev (Triplet) sources.
    """
    text = x["Text"]

    source = x.get("Quadruplet") if "Quadruplet" in x else x.get("Triplet", [])

    answer_list = [f"({t['Aspect']}, {t['Opinion']}, {t['VA']})" for t in source]
    answer = "[Triplet] " + ", ".join(answer_list)

    # Final prompt structure
    prompt = f"{instruction}[Text] {text}\n\nOutput:"

    return {"text": f"<|user|>\n{prompt}\n<|assistant|>\n{answer}"}

In [ ]:
test_sample = all_datasets['laptop']['train'][0]
print(convert(test_sample)['text'])

<|user|>
Below is an instruction describing a task, paired with an input that provides additional context. Your goal is to generate an output that correctly completes the task.

### Instruction:
Given a textual instance [Text], extract all (A, O, VA) triplets, where:
- A is an Aspect term (the entity being discussed)
- O is an Opinion term (the feeling or sentiment expressed about A)
- VA is a Valence–Arousal score in the format (valence#arousal)

Valence: 1.00 (negative) to 9.00 (positive).
Arousal: 1.00 (calm) to 9.00 (excited).
Important: Aspect and Opinion must maintain the exact case and spelling found in the [Text].

### Example:
Input:
[Text] average to good thai food, but terrible delivery.

Output:
[Triplet] (thai food, average to good, 6.75#6.38), (delivery, terrible, 2.88#6.62)

### Question:
Now complete the following example:
Input:
[Text] this unit is ` ` pretty ` ` and stylish , so my high school daughter was attracted to it for that reason .

Output:
<|assistant|>
[Trip

In [ ]:
from datasets import concatenate_datasets

laptop_train_mapped = all_datasets['laptop']['train'].map(
    convert,
    remove_columns=all_datasets['laptop']['train'].column_names
)

restaurant_train_mapped = all_datasets['restaurant']['train'].map(
    convert,
    remove_columns=all_datasets['restaurant']['train'].column_names
)

# Combine for final training set
train_dataset = concatenate_datasets([laptop_train_mapped, restaurant_train_mapped])

#PREPARE VALIDATION SET
laptop_dev_mapped = all_datasets['laptop']['validation'].map(
    convert,
    remove_columns=all_datasets['laptop']['validation'].column_names
)

restaurant_dev_mapped = all_datasets['restaurant']['validation'].map(
    convert,
    remove_columns=all_datasets['restaurant']['validation'].column_names
)

eval_dataset = concatenate_datasets([laptop_dev_mapped, restaurant_dev_mapped])

print(f"Success! Training set: {len(train_dataset)} samples.")
print(f"Success! Validation set: {len(eval_dataset)} samples.")


Map:   0%|          | 0/4076 [00:00<?, ? examples/s]

Map:   0%|          | 0/2284 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Success! Training set: 6360 samples.
Success! Validation set: 400 samples.


### Load the LLM from Hugging Face and apply LoRA for fine-tuning


In [ ]:
# Install Unsloth for Colab
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-az1bdkzi/unsloth_5d4268590df14967873224d888ab8bc1
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-az1bdkzi/unsloth_5d4268590df14967873224d888ab8bc1
  Resolved https://github.com/unslothai/unsloth.git to commit e51d3ea2e498fc893770d92ca6727bd113918480
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 41.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 23.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import get_token
import gc
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

domains = ["laptop", "restaurant"]
model_path = "unsloth/llama-3.1-8b-instruct-bnb-4bit"

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gc.collect()

for domain in domains:
    print(f"\n" + "-"*50)
    print(f"STARTING FINE-TUNING: Llama-3.1-8B ({domain.upper()} DATA)")
    print(f"-"*50)

    clear_vram()
    model = None
    trainer = None

    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = model_path,
            max_seq_length = 1024,
            load_in_4bit = True,
            token = get_token()
        )

        model = FastLanguageModel.get_peft_model(
            model,
            r = 16,
            target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                              "gate_proj", "up_proj", "down_proj",],
            lora_alpha = 16,
            lora_dropout = 0,
            bias = "none",
            use_gradient_checkpointing = "unsloth",
            random_state = 3407,
        )

        current_train = laptop_train_mapped if domain == "laptop" else restaurant_train_mapped
        current_eval = laptop_dev_mapped if domain == "laptop" else restaurant_dev_mapped

        #Trainer Configuration
        trainer = SFTTrainer(
            model = model,
            tokenizer = tokenizer,
            train_dataset = current_train,
            eval_dataset = current_eval,
            dataset_text_field = "text",
            max_seq_length = 1024,
            args = TrainingArguments(
                per_device_train_batch_size = 1,
                gradient_accumulation_steps = 8,
                warmup_steps = 5,
                max_steps = 300,
                learning_rate = 2e-4,
                fp16 = True,
                fp16_full_eval = True,
                per_device_eval_batch_size = 1,
                eval_accumulation_steps = 4,
                logging_steps = 10,
                eval_strategy = "steps",
                eval_steps = 50,
                eval_on_start = True,
                optim = "adamw_8bit",
                output_dir = f"outputs_Llama_{domain}",
                report_to = "none"
            ),
        )

        trainer.train()
        save_path = f"final_adapter_Llama_{domain}"
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"Successfully finished fine-tuning for {domain}!")

    except Exception as e:
        print(f"Error during {domain} training: {e}")

    # Explicit cleanup before next run
    if trainer is not None: del trainer
    if model is not None: del model
    clear_vram()

print("\nAll domain fine-tuning sessions are complete!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

--------------------------------------------------
STARTING FINE-TUNING: Llama-3.1-8B (LAPTOP DATA)
--------------------------------------------------
==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/2284 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/200 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,284 | Num Epochs = 2 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss,Validation Loss
0,No log,2.170696
50,0.203900,0.237000
100,0.224600,0.237667
150,0.204900,0.237007
200,0.176700,0.237936
250,0.173200,0.231936
300,0.178500,0.231021


Successfully finished fine-tuning for restaurant!

All domain fine-tuning sessions are complete!


### Run Inference and Extract Structured Sentiment Outputs

In [ ]:
import re
import json
import torch
from unsloth import FastLanguageModel

# Configuration
domains = ["laptop", "restaurant"]

def extract_triplets(text):
    """Parses (Aspect, Opinion, VA) triplets from model output."""
    result = []
    pattern = r'\(([^,]+),\s*([^,]+),\s*([\d.]+#[\d.]+)\)'
    matches = re.findall(pattern, text)
    for aspect, opinion, va in matches:
        result.append({
            "Aspect": aspect.strip(),
            "Opinion": opinion.strip(),
            "VA": va.strip()
        })
    return result

for domain in domains:
    print(f"\n" + "="*50)
    print(f"LOADING TASK 2 ADAPTER: {domain.upper()}")
    print(f"="*50)

    #Load Model with Adapter
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = f"final_adapter_Llama_{domain}",
        max_seq_length = 1024,
        load_in_4bit = True,
        device_map = {"": 0}
    )

    tokenizer.pad_token = "<|reserved_special_token_0|>"
    model.config.pad_token_id = tokenizer.pad_token_id

    FastLanguageModel.for_inference(model)
    current_test = all_datasets[domain]["test"]
    predictions = []

    print(f"Processing {len(current_test)} samples for {domain}...")

    for i, sample in enumerate(current_test):
        prompt = f"{instruction}[Text] {sample['Text']}\n\nOutput:"

        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to("cuda")

        # Generate Output
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            use_cache=True,
            temperature=0.1,
            eos_token_id=tokenizer.eos_token_id
        )

        input_length = inputs["input_ids"].shape[-1]
        decoded = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        predictions.append({
            "ID": sample.get("ID", f"{domain}_{i}"),
            "Triplet": extract_triplets(decoded)
        })

        if i % 100 == 0:
            print(f"[{domain}] Processed {i}/{len(current_test)} samples...")

    if domain == "laptop":
        laptop_results = predictions
    else:
        restaurant_results = predictions

    del model, tokenizer
    torch.cuda.empty_cache()
    print(f"{domain.capitalize()} Inference Complete. Memory cleared.")

print("\nTask 2 Inference Cycle Finished!")


LOADING TASK 2 ADAPTER: LAPTOP
==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Processing 1000 samples for laptop...
[laptop] Processed 0/1000 samples...
[laptop] Processed 100/1000 samples...
[laptop] Processed 200/1000 samples...
[laptop] Processed 300/1000 samples...
[laptop] Processed 400/1000 samples...
[laptop] Processed 500/1000 samples...
[laptop] Processed 600/1000 samples...
[laptop] Processed 700/1000 samples...
[laptop] Processed 800/1000 samples...
[laptop] Processed 900/1000 samples...
Laptop Inference Complete. Memory cleared.

LOADING TASK 2 ADAPTER: RESTAURANT
==((====))==  U

### Save prediction results

In [ ]:
import json
import os
import zipfile
from google.colab import files

subtask = "subtask_2"
lang = "eng"
domains = ["laptop", "restaurant"]

os.makedirs(subtask, exist_ok=True)

all_results = {
    "laptop": laptop_results,
    "restaurant": restaurant_results
}

for domain, results in all_results.items():
    out_name = f"pred_{lang}_{domain}.jsonl"
    jsonl_path = os.path.join(subtask, out_name)

    print(f"Saving {domain} predictions to {jsonl_path}...")
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for item in results:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

zip_name = f"{subtask}.zip"
print(f"Creating {zip_name}...")

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files_in_dir in os.walk(subtask):
        for file in files_in_dir:
            full_path = os.path.join(root, file)
            zf_path = os.path.relpath(full_path, os.path.dirname(subtask))
            zf.write(full_path, zf_path)

print("Downloading submission file...")
files.download(zip_name)

print("\nAll Done!")

Saving laptop predictions to subtask_2/pred_eng_laptop.jsonl...
Saving restaurant predictions to subtask_2/pred_eng_restaurant.jsonl...
Creating subtask_2.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All Done!


# Subtask 3

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
import os
import re
import json
from datasets import load_dataset

data_urls = {
    "laptop": {
        "train": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_3/eng/eng_laptop_train_alltasks.jsonl",
        "dev": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_3/eng/eng_laptop_dev_task3.jsonl",
        "test": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_3/eng/eng_laptop_test_task3.jsonl"
    }
}

all_datasets_task3 = {}

for domain in ["laptop"]:
    print(f"Loading Subtask 3 data for: {domain.upper()}...")

    all_datasets_task3[domain] = {
        "train": load_dataset("json", data_files=data_urls[domain]["train"], split="train"),
        "dev": load_dataset("json", data_files=data_urls[domain]["dev"], split="train"),
        "test": load_dataset("json", data_files=data_urls[domain]["test"], split="train")
    }

    print(f"{domain.capitalize()} - Train: {len(all_datasets_task3[domain]['train'])}, "
          f"Dev: {len(all_datasets_task3[domain]['dev'])}, "
          f"Test: {len(all_datasets_task3[domain]['test'])}")

print("\nSample Data Format (Laptop Train):")
print(all_datasets_task3["laptop"]["train"][0])

Loading Subtask 3 data for: LAPTOP...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Laptop - Train: 4076, Dev: 200, Test: 1000

Sample Data Format (Laptop Train):
{'ID': 'laptop_quad_dev_1', 'Text': 'this unit is ` ` pretty ` ` and stylish , so my high school daughter was attracted to it for that reason .', 'Quadruplet': [{'Aspect': 'unit', 'Category': 'LAPTOP#DESIGN_FEATURES', 'Opinion': 'pretty', 'VA': '7.12#7.12'}, {'Aspect': 'unit', 'Category': 'LAPTOP#DESIGN_FEATURES', 'Opinion': 'stylish', 'VA': '7.12#7.12'}]}


In [3]:


# SCHEMA DEFINITIONS ---

rest_entity = 'RESTAURANT, FOOD, DRINKS, AMBIENCE, SERVICE, LOCATION'
rest_attribute = 'GENERAL, PRICES, QUALITY, STYLE_OPTIONS, MISCELLANEOUS'

laptop_entity = 'LAPTOP, DISPLAY, KEYBOARD, MOUSE, MOTHERBOARD, CPU, FANS_COOLING, PORTS, MEMORY, POWER_SUPPLY, OPTICAL_DRIVES, BATTERY, GRAPHICS, HARD_DISK, MULTIMEDIA_DEVICES, HARDWARE, SOFTWARE, OS, WARRANTY, SHIPPING, SUPPORT, COMPANY'
laptop_attribute = 'GENERAL, PRICE, QUALITY, DESIGN_FEATURES, OPERATION_PERFORMANCE, USABILITY, PORTABILITY, CONNECTIVITY, MISCELLANEOUS'

def get_instruction(domain):
    entities = laptop_entity if domain == "laptop" else rest_entity
    attributes = laptop_attribute if domain == "laptop" else rest_attribute

    return f'''You are an expert Linguist specializing in Aspect-Based Sentiment Analysis (ABSA). Your task is to extract highly accurate (A, C, O, VA) quadruplets from the given text.

### **Core Extraction Rules:**
1. **Aspect (A):** The specific feature or entity mentioned. Must match the input text casing exactly.
2. **Category (C):** Classify the aspect using the "ENTITY#ATTRIBUTE" schema below. Use ONLY these labels. Must be UPPERCASE.
3. **Opinion (O):** The specific word/phrase used to express the sentiment. Match the input casing exactly.
4. **Valence-Arousal (VA):** - **Valence:** 1.00 (Extremely Negative) to 9.00 (Extremely Positive). 5.00 is Neutral.
   - **Arousal:** 1.00 (Calm/Sleepy) to 9.00 (Excited/Angry). 5.00 is Moderate.
   - Format as "V.VV#A.AA" (always 2 decimal places).

### **Label Schema Constraints:**
- **Valid Entities:** {entities}
- **Valid Attributes:** {attributes}

### **Step-by-Step Reasoning:**
Step 1: Identify all sentiment-bearing phrases and the aspects they refer to.
Step 2: Map each aspect to the most relevant ENTITY and ATTRIBUTE from the schema.
Step 3: Determine the numerical Valence (positivity) and Arousal (intensity) of the opinion.
Step 4: Format as a list of (A, C, O, VA) quadruplets.

---
### **Examples:**

**Input:** [Text] The screen is incredibly bright and vibrant, but the price is a bit steep.
**Output:** [Quadruplet] (screen, DISPLAY#QUALITY, incredibly bright and vibrant, 8.50#7.20), (price, LAPTOP#PRICE, a bit steep, 3.20#5.50)

**Input:** [Text] The waiter was polite but the food arrived cold.
**Output:** [Quadruplet] (waiter, SERVICE#GENERAL, polite, 7.00#4.50), (food, FOOD#QUALITY, cold, 2.50#6.80)

---
### **Target Task:**
Input:
[Text] {{input_text}}

Output:
'''



In [4]:
def convert_to_quads(x, domain):
    text = x["Text"]
    source = x.get("Quadruplet", [])

    # Format (Aspect, Category, Opinion, VA)
    quad_strings = [f"({q['Aspect']}, {q['Category']}, {q['Opinion']}, {q['VA']})" for q in source]
    answer = "[Quadruplet] " + ", ".join(quad_strings)

    # Build Prompt
    prompt = f"{get_instruction(domain)}{text}\n\nOutput:"

    return {"text": f"<|user|>\n{prompt}\n<|assistant|>\n{answer}<|end_of_text|>"}


# Process datasets
processed_datasets = {}
for domain in ["laptop"]:
    print(f"Loading and Mapping {domain.upper()}...")
    train_raw = load_dataset("json", data_files=data_urls[domain]["train"], split="train")
    dev_raw = load_dataset("json", data_files=data_urls[domain]["dev"], split="train")

    processed_datasets[domain] = {
        "train": train_raw.map(lambda x: convert_to_quads(x, domain), remove_columns=train_raw.column_names),
        "dev": dev_raw.map(lambda x: convert_to_quads(x, domain), remove_columns=dev_raw.column_names)
    }

Loading and Mapping LAPTOP...


Map:   0%|          | 0/4076 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [5]:
import os
import torch
import gc
import re
import json
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import get_token

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

model_path = "unsloth/llama-3.1-8b-instruct-bnb-4bit"

for domain in ["laptop"]:
    print(f"\nSTARTING TASK 3 TRAINING: {domain.upper()}")
    clear_vram()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_path,
        max_seq_length = 1024,
        load_in_4bit = True,
        token = get_token()
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        use_gradient_checkpointing = "unsloth",
    )

    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = processed_datasets[domain]["train"],
        eval_dataset = processed_datasets[domain]["dev"],
        dataset_text_field = "text",
        max_seq_length = 1024,
        args = TrainingArguments(
            per_device_train_batch_size = 1,
            gradient_accumulation_steps = 8,
            max_steps = 500,
            learning_rate = 2e-4,
            fp16 = True,
            fp16_full_eval = True,
            logging_steps = 20,
            eval_strategy = "steps",
            eval_steps = 50,
            torch_compile = True,
            optim = "adamw_8bit",
            output_dir = f"outputs_task3_{domain}",
            report_to = "none"
        ),
    )

    trainer.train()

    # Task 3 Adapters
    model.save_pretrained(f"task3_adapter_{domain}")
    tokenizer.save_pretrained(f"task3_adapter_{domain}")

    # Clean up before next domain
    del model, trainer
    clear_vram()

print("\nSubtask 3 Fine-Tuning Complete for all domains!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

STARTING TASK 3 TRAINING: LAPTOP
==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,076 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Enabled auto compiling


Step,Training Loss,Validation Loss
50,0.103300,0.105609
100,0.097200,0.104313
150,0.093800,0.097448
200,0.092200,0.105125
250,0.091800,0.103395
300,0.090500,0.104215
350,0.086600,0.107782
400,0.091900,0.099530
450,0.085600,0.101508
500,0.090500,0.100830


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



Subtask 3 Fine-Tuning Complete for all domains!


In [6]:
import re
import json
import torch
from unsloth import FastLanguageModel

def extract_quadruplets(text):
    result = []
    pattern = r'\(([^,]+),\s*([^,]+),\s*([^,]+),\s*([\d.]+#[\d.]+)\)'
    matches = re.findall(pattern, text)

    for aspect, category, opinion, va in matches:
        if "NULL" in aspect.upper() or "NULL" in opinion.upper():
            continue

        result.append({
            "Aspect": aspect.strip(),
            "Category": category.strip().upper(),
            "Opinion": opinion.strip(),
            "VA": va.strip()
        })
    return result

#Sequential Prediction Loop
all_task3_results = {}

for domain in ["laptop"]:
    print(f"\n" + "="*50)
    print(f"RUNNING TASK 3 INFERENCE: {domain.upper()}")
    print(f"="*50)

    # Load the domain-specific adapter
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = f"task3_adapter_{domain}",
        max_seq_length = 1024,
        load_in_4bit = True,
        device_map = {"": 0}
    )
    FastLanguageModel.for_inference(model)
    domain_instruction = get_instruction(domain)

    test_data = all_datasets_task3[domain]["test"]
    predictions = []

    for i, sample in enumerate(test_data):
        prompt = f"{domain_instruction}{sample['Text']}\n\nOutput:"

        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(inputs, return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            use_cache=True,
            temperature=0.1,
            eos_token_id=tokenizer.eos_token_id
        )

        # Decode Assistant response only
        input_len = inputs["input_ids"].shape[-1]
        decoded = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

        # Structure the final JSON
        dump_data = {
            "ID": sample["ID"],
            "Quadruplet": extract_quadruplets(decoded)
        }

        predictions.append(dump_data)

        if i % 50 == 0:
            print(f"[{domain}] {i}/{len(test_data)} processed.")

    all_task3_results[domain] = predictions

    # Clear memory for next domain
    del model, tokenizer
    torch.cuda.empty_cache()

print("\nTask 3 Extraction Complete!")


RUNNING TASK 3 INFERENCE: LAPTOP
==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
[laptop] 0/1000 processed.
[laptop] 50/1000 processed.
[laptop] 100/1000 processed.
[laptop] 150/1000 processed.
[laptop] 200/1000 processed.
[laptop] 250/1000 processed.
[laptop] 300/1000 processed.
[laptop] 350/1000 processed.
[laptop] 400/1000 processed.
[laptop] 450/1000 processed.
[laptop] 500/1000 processed.
[laptop] 550/1000 processed.
[laptop] 600/1000 processed.
[laptop] 650/1000 processed.
[laptop] 700/1000 processed.
[laptop] 750/1000 processed.
[laptop] 800/1000 processed.
[laptop] 850/1000 processed.

In [7]:
import json
import os
import zipfile
from google.colab import files

# Submission Configuration
subtask = "subtask_3"
lang = "eng"
domains = ["laptop"]

os.makedirs(subtask, exist_ok=True)


for domain in domains:
    # Resolve official filename
    out_name = f"pred_{lang}_{domain}.jsonl"
    jsonl_path = os.path.join(subtask, out_name)

    print(f"Writing {domain} predictions to {jsonl_path}...")

    results_list = all_task3_results.get(domain, [])

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for item in results_list:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

zip_name = f"{subtask}.zip"
print(f"Creating {zip_name}...")

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files_in_dir in os.walk(subtask):
        for file in files_in_dir:
            full_path = os.path.join(root, file)
            zf_path = os.path.relpath(full_path, ".")
            zf.write(full_path, zf_path)

# Browser download
print("Downloading submission file...")
files.download(zip_name)

print("\nSubtask 3 submission is ready! Good luck on the leaderboard!")

Writing laptop predictions to subtask_3/pred_eng_laptop.jsonl...
Creating subtask_3.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Subtask 3 submission is ready! Good luck on the leaderboard!
